In [2]:
!pip install mediapipe pandas numpy scipy

   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/10.2 MB 11.2 MB/s eta 0:00:01
   ------------ --------------------------- 3.1/10.2 MB 10.9 MB/s eta 0:00:01
   ---------------------- ----------------- 5.8/10.2 MB 10.7 MB/s eta 0:00:01
   ------------------------------- -------- 8.1/10.2 MB 10.7 MB/s eta 0:00:01
   ---------------------------------------- 10.2/10.2 MB 10.6 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.2.10
    Uninstalling flatbuffers-25.2.10:
      Successfully uninstalled flatbuffers-25.2.10


In [2]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from collections import deque, Counter
from scipy.interpolate import interp1d
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


def normalize_skeleton(data):
    if np.max(np.abs(data)) < 1e-6:
        return data
    hip = data[:, 0, :, :].copy()
    data = data - hip[:, np.newaxis, :, :]
    scale = np.max(np.abs(data))
    if scale > 1e-6:
        data = data / scale
    return data


def interpolate_frames(data, target=30):
    T, M, V, C = data.shape
    if T == target:
        return data
    old_t, new_t = np.linspace(0, 1, T), np.linspace(0, 1, target)
    result = np.zeros((target, M, V, C), dtype=np.float32)
    for m in range(M):
        for v in range(V):
            for c in range(C):
                f = interp1d(old_t, data[:, m, v, c], kind='linear', 
                          bounds_error=False, 
                          fill_value=(data[0, m, v, c], data[-1, m, v, c]))
                result[:, m, v, c] = f(new_t)
    return result


class LSTMSkeletonNet(nn.Module):
    def __init__(self, num_classes=60, input_size=75, hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # LSTM принимает (batch, seq_len, input_size)
        self.lstm = nn.LSTM(
            input_size=input_size,     # 25 суставов × 3 координаты = 75
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Вход: (N, T, C, V) → хотим (N, T, C*V)
        N, T, C, V = x.shape
        x = x.view(N, T, -1)  # (N, 30, 75)

        # LSTM
        lstm_out, (hidden, _) = self.lstm(x)  # hidden: (num_layers, N, hidden_size)

        # Берём последнее состояние
        out = hidden[-1]  # (N, hidden_size)

        # Классификация
        return self.classifier(out)


CLASSES = [
    'drink water', 'eat meal', 'brushing teeth', 'brushing hair', 'drop', 
    'pickup', 'throw', 'sitting down', 'standing up', 'clapping', 'reading', 
    'writing', 'tear up paper', 'wear jacket', 'taking off jacket', 
    'wear a shoe', 'taking off a shoe', 'wear socks', 'taking off socks', 
    'stretching arm', 'kicking', 'punching', 'kicking 2', 'punching 2', 
    'falling', 'hammering', 'kicking something', 'punching 3', 'dancing', 
    'kicking 3', 'writing 2', 'taking a selfie', 'checking time', 
    'rub two hands together', 'walking zigzag', 'walking with irregular speed', 
    'walking with heavy steps', 'arm circles', 'arm swings', 'lunge', 
    'squats', 'banded squats', 'arm curls', 'prior box squats', 'pushups', 
    'bench press', 'deadlift', 'jump jacks', 'rowing', 'running on treadmill', 
    'situps', 'lunges', 'jump rope', 'pushup jacks', 'high knees', 
    'heels down', 'side kick', 'round house kick', 'fore kick', 'side kick 2', 
    'side lunge'
]


class MovieActionAnalyzer:
    def __init__(self, model_path='models/best_model.pth', 
                 pose_model_path='pose_landmarker_full.task'):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"🔧 Устройство: {self.device}")
        
        self.model = LSTMSkeletonNet(num_classes=60).to(self.device)
        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device, weights_only=True)
        )
        self.model.eval()
        print(f"✅ Модель загружена")
        
        if not os.path.exists(pose_model_path):
            raise FileNotFoundError(f"Pose model не найдена: {pose_model_path}")
        
        base_options = python.BaseOptions(model_asset_path=pose_model_path)
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.pose = vision.PoseLandmarker.create_from_options(options)
        print("✅ Pose Landmarker инициализирован")
        
        self.skeleton_buffer = deque(maxlen=30)
        
        # 🔥 НОВОЕ: буфер для усреднения предсказаний
        self.prediction_buffer = deque(maxlen=5)  # Последние 5 предсказаний
        self.confidence_threshold = 0.15  # Минимальная уверенность (15%)
        
    def mediapipe_to_ntu(self, landmarks):
        if landmarks is None:
            return None
        coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])
        ntu_joints = coords[:25]
        skeleton = ntu_joints[np.newaxis, ...]
        skeleton = np.repeat(skeleton, 2, axis=0)
        return skeleton.transpose(1, 0, 2)
    
    def preprocess(self, skeleton):
        if skeleton is None:
            return None
        data = normalize_skeleton(skeleton)
        data = interpolate_frames(data, target=30)
         # Усредняем по телам → (30, 25, 3), есть над чем поработать, добавит ошибок, если несколько тел
        #data = data.mean(axis=1)  # или data[:, 0, ...] для первого тела
        tensor = torch.FloatTensor(data).permute(2, 0, 1, 3)  # (N, T, C, V)
        return tensor.to(self.device)
    
    def predict(self, frame):
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        results = self.pose.detect(mp_image)
        
        if results.pose_landmarks and len(results.pose_landmarks) > 0:
            skeleton = self.mediapipe_to_ntu(results.pose_landmarks[0])
            if skeleton is not None:
                self.skeleton_buffer.append(skeleton)
        
        if len(self.skeleton_buffer) == 30:
            data = np.stack(list(self.skeleton_buffer), axis=0)
            self.skeleton_buffer.popleft()
            processed = self.preprocess(data)
            if processed is not None:
                with torch.no_grad():
                    outputs = self.model(processed)
                    probs = torch.softmax(outputs, dim=1)[0]
                    
                    # 🔥 НОВОЕ: Берём топ-3 класса
                    top_probs, top_indices = torch.topk(probs, 3)
                    
                    # 🔥 НОВОЕ: Усредняем с предыдущими предсказаниями
                    current_action = CLASSES[top_indices[0].item()]
                    current_conf = top_probs[0].item()
                    
                    self.prediction_buffer.append((current_action, current_conf))
                    
                    # 🔥 НОВОЕ: Голосование по последним предсказаниям
                    actions = [p[0] for p in self.prediction_buffer]
                    confs = [p[1] for p in self.prediction_buffer]
                    
                    # Самый частый класс
                    action_counter = Counter(actions)
                    final_action = action_counter.most_common(1)[0][0]
                    
                    # Средняя уверенность для этого класса
                    avg_conf = np.mean([c for a, c in zip(actions, confs) if a == final_action])
                    
                    # 🔥 НОВОЕ: Порог уверенности
                    if avg_conf < self.confidence_threshold:
                        return "unknown", 0.0, True
                   
                    return final_action, avg_conf, True
        
        return "unknown", 0.0, False
    
    def close(self):
        self.pose.close()


def analyze_movie(video_path, output_csv='movie_analysis.csv', skip_frames=10):
    model_path = 'models/best_model.pth'
    pose_model_path = 'pose_landmarker_full.task'
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Модель не найдена: {model_path}")
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Видео не найдено: {video_path}")
    if not os.path.exists(pose_model_path):
        raise FileNotFoundError(f"Pose model не найдена: {pose_model_path}")
    
    print(f"🎬 Видео: {video_path}")
    print(f"📁 Модель: {model_path}")
    
    analyzer = MovieActionAnalyzer(model_path, pose_model_path)
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        raise ValueError(f"Не удалось открыть видео: {video_path}")
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_min = total_frames / fps / 60
    
    print(f"\n📊 Видео: {duration_min:.1f} мин, {fps:.1f} fps, {total_frames} кадров")
    print(f"📊 Пропуск кадров: {skip_frames}")
    print(f"⏱️  Ожидаемое время: ~{duration_min * 0.2:.0f} мин\n")
    
    results = []
    frame_idx = 0
    poses_detected = 0
    
    print("🔄 Обработка...")
    print("-" * 80)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_idx % skip_frames == 0:
            action, confidence, pose_detected = analyzer.predict(frame)
            if pose_detected:
                poses_detected += 1
            
            timestamp_min = frame_idx / fps / 60
            progress = frame_idx / total_frames * 100
            
            results.append({
                'frame': frame_idx,
                'time_min': round(timestamp_min, 1),
                'time_hms': f"{int(timestamp_min//60):02d}:{int((timestamp_min%60)):02d}",
                'action': action,
                'confidence': f"{confidence:.1%}" if confidence > 0 else "N/A"
            })
            
            if len(results) % 50 == 0:
                print(f"{progress:5.1f}% | {timestamp_min:6.1f}min | {action:20s} | "
                      f"Conf: {confidence:.1%} | Поз: {poses_detected}")
        
        frame_idx += 1
    
    cap.release()
    analyzer.close()
    
    print("-" * 80)
    print(f"\n✅ Готово! Обработано кадров: {frame_idx}, Поз обнаружено: {poses_detected}")
    
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"💾 Результаты: {output_csv}")
    
    if len(df) > 0:
        print(f"\n📊 Распределение действий:")
        action_counts = df['action'].value_counts()
        for action, count in action_counts.head(10).items():
            pct = count / len(df) * 100
            print(f"   {action:25s}: {count:4d} ({pct:5.1f}%)")
    
    return df


if __name__ == "__main__":
    os.makedirs('results', exist_ok=True)
    
    video_file = 'video_samples/potter1 01.mp4'
    if not os.path.exists(video_file):
        print(f"❌ Видео '{video_file}' не найдено!")
    else:
        df_results = analyze_movie(
            video_file, 
            'results/movie_analysis.csv',
            skip_frames=10  # Увеличено для скорости
        )
        print(f"\n🎉 Готово: results/movie_analysis.csv")

🎬 Видео: video_samples/potter1 01.mp4
📁 Модель: models/best_model.pth
🔧 Устройство: cuda
✅ Модель загружена
✅ Pose Landmarker инициализирован

📊 Видео: 3.7 мин, 25.0 fps, 5509 кадров
📊 Пропуск кадров: 10
⏱️  Ожидаемое время: ~1 мин

🔄 Обработка...
--------------------------------------------------------------------------------
  8.9% |    0.3min | unknown              | Conf: 0.0% | Поз: 0
 18.0% |    0.7min | reading              | Conf: 70.7% | Поз: 47
 27.0% |    1.0min | unknown              | Conf: 0.0% | Поз: 85
 36.1% |    1.3min | wear jacket          | Conf: 84.7% | Поз: 117
 45.2% |    1.7min | reading              | Conf: 93.8% | Поз: 160
 54.3% |    2.0min | wear jacket          | Conf: 86.9% | Поз: 195
 63.4% |    2.3min | side kick 2          | Conf: 73.5% | Поз: 240
 72.4% |    2.7min | standing up          | Conf: 68.6% | Поз: 286
 81.5% |    3.0min | wear jacket          | Conf: 49.4% | Поз: 336
 90.6% |    3.3min | reading              | Conf: 56.6% | Поз: 384
-------